# 1-board: sample-timing

Find from which duty each crest sample reads the drive honestly, and where in a short window it lands, so the floors the firmware publishes are graded against the board they run on.

## Purpose

The firmware samples the shunt and the taps once per PWM period, at fixed instants after the counter's crest, and reads them only when the drive window is at least `i_window_min_ticks` or `v_window_min_ticks` wide. Below those floors the current limit, calibration and identification are blind. On a resistor the current inside a window is the same at every duty, so a rung whose crest sample reads less than the 100 % rung reads early, and the lowest honest rung is the floor. The bursts draw the current and the terminals inside one period at a resolution the crest sample never has, which says where the edges and the settle are.

## Procedure

A duty ladder on the bare load at 7.4 V, 3 to 40 % and the 100 % control, both directions, in slow decay and again in fast, each rung 150 ms with a rest between. Then `osc ident burst` with the static load at four duties, three captures each, with the shunt and both taps interleaved.

1. The board on the DPS, the bare load on the terminals, the adapter on the bus at 3 Mbps. No meter.
2. `capture.py`. It raises the current limit for the ladders and puts the saved one back.

## Data

- `ladder-slow/`, `ladder-fast/`: `sweep.csv` and `meta.json`, one row per control tick, one segment per rung.
- `bursts/burst-<k>.csv`: the free-running ADC through PWM periods around a duty step; `code` is the raw sample, the meta fields on the first row.

### The ladders

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oscnb import b41t, boards, display, laps, results, tel
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"
BOARD = boards.BOARDS[boards.DEFAULT]
CAPTURE = Path("../../../telemetry") / BOARD.key / "sample-timing" / "capture-1"
CAPTURE = CAPTURE.resolve()
print(BOARD.key, "from", CAPTURE)

In [ ]:
from oscnb import bursts as bu
ladders = {d: tel.load(CAPTURE / f"ladder-{d}", BOARD) for d in ("slow", "fast") if (CAPTURE / f"ladder-{d}" / "sweep.csv").exists()}
fig, ax = plt.subplots(len(ladders), 1, figsize=(12, 3.5 * len(ladders)), squeeze=False)
for a, (d, (df, meta)) in zip(ax[:, 0], ladders.items()):
    a.plot(df["t"], df["current_raw"], lw=0.3, color=BLUE); a.set_ylabel("current_raw"); a.set_title(f"{d} decay"); a.grid(alpha=0.3)
    print(f"{d}: {df['seg'].max()} rungs, gaps {tel.gaps(df).to_dict() or 'none'}")
ax[-1, 0].set_xlabel("time (s)")
plt.show()

The current channel through each ladder: the rest, then every rung of each direction. On a resistor every honest rung sits at the same level as the 100 % rung, less the rail's sag.

## Method

Each rung's settled mean of the current channel over its rest zero, and of the tap difference, as a ratio to the 100 % rung of the same direction. The rail sags with the mean current, so each ratio is corrected by the rail tap's own ratio first. Walking down from 40 %, the lowest rung with every rung above it inside 1 % is the 1 % floor, and the same at 3 %. Walking down matters: a rung that reads early reads low, and a one-sided test would pass it. The floors are compared with the ones the firmware publishes in the registry.

### Rungs against the 100 % rung

In [ ]:
rows = []
for d, (df, meta) in ladders.items():
    zero = tel.baseline(df)["current_raw"].median()
    r = tel.rungs(df)
    r["i"] = r["current_raw"] - zero; r["v"] = (r["vmotor_a"] - r["vmotor_b"]).abs(); r["duty_pct"] = BOARD.duty_pct_of_ticks(r["duty_ticks"])
    for sgn, g in r.groupby("dir"):
        ref = g[g["cmd_duty_q15"].abs() >= 32000]
        if not len(ref):
            continue
        ref = ref.iloc[0]
        rail = g["vbus_raw"] / ref["vbus_raw"]
        g = g.assign(decay=d, i_ratio=g["i"] / ref["i"] / rail, v_ratio=g["v"] / ref["v"] / rail)
        rows.append(g)
R = pd.concat(rows, ignore_index=True)
show(R[["decay", "dir", "cmd_duty_q15", "duty_pct", "duty_ticks", "i", "v", "i_ratio", "v_ratio", "n"]], n=None)

One row per rung, decay and direction: its current and tap difference in counts, and each as a ratio to the 100 % rung after the rail's sag is taken out.

In [ ]:
def floor_pct(g, col, tol):
    # walk down from the top rung; the floor is the lowest rung with every rung above it inside the band
    g = g[g["cmd_duty_q15"].abs() < 32000].sort_values("duty_pct", ascending=False)
    low = np.nan
    for pct, good in zip(g["duty_pct"], (g[col] - 1).abs() <= tol):
        if not good:
            break
        low = pct
    return low
floors = []
for (d, sgn), g in R.groupby(["decay", "dir"]):
    floors.append({"decay": d, "dir": sgn, "i_1pct": floor_pct(g, "i_ratio", 0.01), "i_3pct": floor_pct(g, "i_ratio", 0.03),
                   "v_1pct": floor_pct(g, "v_ratio", 0.01), "v_3pct": floor_pct(g, "v_ratio", 0.03)})
F = pd.DataFrame(floors)
show(F, n=None)
print(f"published: current from {BOARD.i_floor_pct:.1f} % ({BOARD.i_window_min_ticks} ticks), terminals from {BOARD.v_floor_pct:.1f} % ({BOARD.v_window_min_ticks} ticks)")
fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for a, col, name in ((ax[0], "i_ratio", "current"), (ax[1], "v_ratio", "terminal difference")):
    for (d, sgn), g in R[R["cmd_duty_q15"].abs() < 32000].groupby(["decay", "dir"]):
        a.plot(g["duty_pct"], (g[col] - 1) * 100, marker="o", lw=0.8, ms=4, color=BLUE if d == "slow" else ORANGE, ls="-" if sgn > 0 else "--", label=f"{d} {sgn:+d}")
    a.axhspan(-3, 3, color=MUTED, alpha=0.15); a.axhspan(-1, 1, color=MUTED, alpha=0.15)
    a.axvline(BOARD.i_floor_pct if col == "i_ratio" else BOARD.v_floor_pct, color=INK, lw=0.6, ls=":")
    a.set_xlabel("duty (%)"); a.set_title(f"{name} against the 100 % rung"); a.grid(alpha=0.3); a.legend(fontsize=8)
ax[0].set_ylabel("error (%)")
plt.show()

Each rung's error against the 100 % rung, current on the left and terminal difference on the right, with the 1 % and 3 % bands and the published floor as the dotted line. A channel is honest from the duty where its curve enters the band and stays.

### Inside one period

In [ ]:
caps = bu.load(CAPTURE / "bursts")
print(len(caps), "bursts")
arr = BOARD.pwm_arr
widest = max(caps, key=lambda c: abs(bu.meta(c)["step_q15"]))
m = bu.meta(widest); half = int(BOARD.drive_ticks(m["step_q15"]))
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
settle = {}
for a, (stream, name) in zip(ax, (("sh", "shunt"), ("ta", "tap A"), ("tb", "tap B"))):
    t, y, _ = bu.fold(widest, arr, stream)
    if not len(t):
        a.set_title(f"{name}: not in the burst"); continue
    x, prof = bu.profile(t, y, -arr, arr, 4)
    a.plot(x, prof, lw=0.8, color=BLUE); a.axvspan(-half, half, color=ORANGE, alpha=0.12); a.set_xlabel("ticks from the crest"); a.set_title(f"{name} inside the {m['step_q15'] * 100 / 32768:.0f} % window"); a.grid(alpha=0.3)
    if stream == "sh":
        z = bu.zero(widest, arr)
        plateau = np.median(prof[(x > half - 200) & (x < half - 20)])
        inside = (x >= -half) & (x <= half)
        within = np.abs(prof - z) >= 0.99 * abs(plateau - z)
        first = x[inside & within]
        settle["shunt_settle_ticks"] = float(first[0] + half) if len(first) else np.nan
        a.axhline(plateau, color=INK, lw=0.5, ls=":")
        print(f"shunt: zero {z:.0f}, plateau {plateau:.0f} counts, inside 1 % from {settle['shunt_settle_ticks']:.0f} ticks after the window opens; the crest sample closes {half + 31} ticks after it")
plt.show()

Each stream folded onto the period of the widest burst, the drive window shaded. The current climbs from the window's start and settles on its plateau; the time it takes to come inside 1 % is the shunt's settle, and a window shorter than that plus the sample's own offset from the crest reads early.

## Results

The floors from the ladders per channel and tolerance in slow decay, the shunt's settle from the fold, and the published floors for comparison.

In [ ]:
slow = F[F["decay"] == "slow"] if (F["decay"] == "slow").any() else F
results.write("1-board", "sample-timing", CAPTURE, pair=BOARD.key,
              notes="duty ladders on the bare load against the 100 % rung, both decays and directions; bursts folded on the PWM period",
              i_floor_1pct=results.quantity(slow["i_1pct"].max(), lo=slow["i_1pct"].min(), hi=slow["i_1pct"].max(), unit="% duty"),
              i_floor_3pct=results.quantity(slow["i_3pct"].max(), lo=slow["i_3pct"].min(), hi=slow["i_3pct"].max(), unit="% duty"),
              v_floor_1pct=results.quantity(slow["v_1pct"].max(), lo=slow["v_1pct"].min(), hi=slow["v_1pct"].max(), unit="% duty"),
              v_floor_3pct=results.quantity(slow["v_3pct"].max(), lo=slow["v_3pct"].min(), hi=slow["v_3pct"].max(), unit="% duty"),
              published_i_floor=results.quantity(BOARD.i_floor_pct, unit="% duty"),
              published_v_floor=results.quantity(BOARD.v_floor_pct, unit="% duty"),
              **({"shunt_settle_ticks": results.quantity(settle["shunt_settle_ticks"], unit="ticks")} if settle else {}))

## Limitations

- A resistor has no inductance, so the current steps on at the window's edge and this is the worst case for the shunt's settle. A winding's current ramps instead, which is easier on the sample and harder on the reading's meaning.
- One load and one rail. The settle is the amplifier's and should not change with the current, but that is an assumption until a smaller load is folded the same way.
- The ladder's lowest rung is 3 %; a floor under it reads as 3 %.

## Open questions

- Whether the settle scales with the step. The same bursts at 10 to 25 % duty on the series load, a third of the current, folded onto this one would say.

## Conclusion

To be written from the capture.